# Week 10: FitPulse Streaming Simulation

**Project:** FitPulse Wellness Analytics Hub  
**Team:** Team 11  
**Notebook:** `07_streaming_simulation`

## Objective

Document the Week 10 streaming exercises: incremental JSON ingestion with Auto Loader and a simulated live workout-event flow using Delta tables and Structured Streaming.

**Important:** The ingestion runs described below were performed in Databricks. The validation cells in this GitHub notebook are read-only checks; they do not rerun ingestion.

## Demo 1: Auto Loader JSON file ingestion

### Configuration

- **Input directory:** `/Volumes/fitpulse-wellness/default/fitpulsehub/week10_streaming/drops/`
- **Input files:** `drop_01.json`, `drop_02.json`, `drop_03.json`, `drop_04.json`
- **Target table:** `fitpulse-wellness.default.bronze_workout_event_files`
- **Checkpoint:** `/Volumes/fitpulse-wellness/default/fitpulsehub/week10_streaming/checkpoints/autoloader`
- **Schema location:** `/Volumes/fitpulse-wellness/default/fitpulsehub/week10_streaming/schemas/autoloader`

Auto Loader was used to ingest controlled JSON file drops incrementally into the Bronze table. A persistent checkpoint and schema location were configured for the ingestion run.

### Previously verified results

| Validation | Result |
|---|---:|
| Total rows ingested | 7 |
| Distinct event IDs | 7 |
| Source files | 4 |
| Rescued rows | 0 |

These values are recorded from the Databricks checks performed during the exercise. They are documentation, not newly generated notebook output.

In [ ]:
# Read-only validation for Demo 1.
# This reads the existing Delta table; it does not run Auto Loader.

auto_loader_table = "fitpulse-wellness.default.bronze_workout_event_files"
auto_loader_df = spark.table(auto_loader_table)

total_rows = auto_loader_df.count()
distinct_event_ids = auto_loader_df.select("event_id").distinct().count()

print(f"Target table: {auto_loader_table}")
print(f"Total rows: {total_rows}")
print(f"Distinct event IDs: {distinct_event_ids}")

## Demo 2: Simulated live workout events

### Configuration

- **Source table:** `fitpulse-wellness.default.fitpulse_live_events`
- **Streaming Bronze target:** `fitpulse-wellness.default.bronze_fitpulse_live_events`
- **Checkpoint:** `/Volumes/fitpulse-wellness/default/fitpulsehub/week10_streaming/checkpoints/live_bronze`
- **Trigger used:** `availableNow=True`

A small Python simulator appended synthetic workout events to the source Delta table. Structured Streaming then processed the available source data into the Bronze target using a persistent checkpoint.

### Previously verified results

| Validation | Result |
|---|---:|
| Source events | 7 |
| Distinct source event IDs | 7 |
| Streaming Bronze events | 7 |
| Distinct Bronze event IDs | 7 |

A source-versus-target reconciliation also returned 7 source events and 7 Bronze events.

### Trigger limitation

The cluster rejected the `processingTime="5 seconds"` trigger with `INFINITE_STREAMING_TRIGGER_NOT_SUPPORTED`. The supported `availableNow=True` trigger was used instead. This processes the currently available data and then terminates; it does **not** demonstrate a continuously running stream.

In [ ]:
# Read-only reconciliation for Demo 2.
# This checks the current source and target tables without starting a stream.

live_source_table = "fitpulse-wellness.default.fitpulse_live_events"
live_target_table = "fitpulse-wellness.default.bronze_fitpulse_live_events"

source_df = spark.table(live_source_table)
target_df = spark.table(live_target_table)

source_count = source_df.count()
source_distinct_ids = source_df.select("event_id").distinct().count()
target_count = target_df.count()
target_distinct_ids = target_df.select("event_id").distinct().count()

print(f"Source table: {live_source_table}")
print(f"Source events: {source_count}")
print(f"Distinct source event IDs: {source_distinct_ids}")
print()
print(f"Bronze target: {live_target_table}")
print(f"Bronze events: {target_count}")
print(f"Distinct Bronze event IDs: {target_distinct_ids}")

## Continuous SQL pipeline: Status and blocker

The requested continuous SQL pipeline using `FROM STREAM(...)` has **not been successfully created or verified**.

During pipeline setup, the destination catalog defaulted to `workspace`. Selecting `fitpulse-wellness` produced a permissions error stating that schema creation was not permitted in that catalog.

The continuous pipeline therefore remains pending. The successful `availableNow=True` Structured Streaming run is separate evidence and must not be presented as proof that the continuous SQL pipeline is running.

The continuous pipeline acceptance checks still to be demonstrated are:

1. Create and start the SQL pipeline with the required permissions and destination configuration.
2. Show the Bronze event count increasing while the producer is running.
3. Show the count stabilizing after the producer stops.
4. Restart the producer and demonstrate that new events are processed.

Do not mark these checks complete until they have actually been performed.

## Evidence and completion checklist

- [x] Auto Loader JSON file-drop exercise completed in Databricks.
- [x] Auto Loader results reconciled: 7 rows, 7 distinct event IDs, 4 source files, 0 rescued rows.
- [x] Simulated live-event exercise completed using `availableNow=True`.
- [x] Live source and Bronze target counts reconciled at 7 events each.
- [x] Streaming design document and event-schema contract prepared separately.
- [ ] Continuous SQL pipeline successfully created and verified.
- [ ] Screenshots captured and added under `screenshots/week10_*`.
- [ ] Final notebook export and GitHub commit verified.

## Scope notes

The events used in the simulation are synthetic. The Kafka schema file defines an event contract only; Kafka infrastructure was not deployed as part of this exercise. No screenshots or pipeline execution results are claimed by this notebook unless separately captured and committed.